In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
def extract_cls_embeddings(texts, tokenizer, model, batch_size=16):
    embeddings = []
    model.eval()
    device = model.device  # 모델이 할당된 디바이스 가져오기
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting [CLS] embeddings"):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: val.to(device) for key, val in inputs.items()}  # 모든 입력을 모델의 디바이스로 이동
            outputs = model(**inputs)
            cls_batch = outputs.last_hidden_state[:, 0, :]
            embeddings.append(cls_batch.cpu().numpy())  # GPU 메모리를 절약하기 위해 CPU로 이동 후 numpy 변환
    return np.vstack(embeddings)

In [ ]:
def extract_first_image_urls(images_list):
    # 비어있거나 잘못된 값 방지
    if not images_list or not isinstance(images_list, list) or len(images_list) == 0:
        return pd.Series([None, None, None])
    img = images_list[0]
    return pd.Series([
        img.get('small_image_url'),
        img.get('medium_image_url'),
        img.get('large_image_url')
    ])

## BERT

In [ ]:
# 데이터 로딩
df = pd.read_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v2.json')
print(df.shape)
df.head()

In [ ]:
# 사용자, 아이템별 리뷰 집합 생성
user_review_set = df.groupby('user_id')['text'].apply(lambda row: ' '.join(row)).reset_index()
item_review_set = df.groupby('asin')['text'].apply(lambda row: ' '.join(row)).reset_index()

# column 변경
user_review_set.columns = ['user_id', 'user_text']
item_review_set.columns = ['asin', 'item_text']

# 결과 확인
print(user_review_set.shape)
print(item_review_set.shape)

user_review_set.head()

In [ ]:
item_review_set.head()

In [ ]:
# user, item 수 확인
user_review_set.shape, item_review_set.shape

In [ ]:
# load bert
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

In [ ]:
# 사용자 리뷰 집합에서 bert embedding 추출 후 concat
user_text_embeddings = extract_cls_embeddings(user_review_set['user_text'].tolist(), tokenizer, model, batch_size=16)
user_text_embeddings = pd.DataFrame({'user_bert_emb': [e for e in user_text_embeddings]})
user_review_set = pd.concat([user_review_set, user_text_embeddings], axis=1)

In [ ]:
# 아이템 리뷰 집합에서 bert embedding 추출 후 concat
item_text_embeddings = extract_cls_embeddings(item_review_set['item_text'].tolist(), tokenizer, model, batch_size=16)
item_text_embeddings = pd.DataFrame({'item_bert_emb': [e for e in item_text_embeddings]})
item_review_set = pd.concat([item_review_set, item_text_embeddings], axis=1)

In [ ]:
# 뽑은 aspect를 join
df = pd.merge(df, user_review_set, on='user_id')
df = pd.merge(df, item_review_set, on='asin')
df= df.drop(columns=['user_text', 'item_text'])

In [ ]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v3.pkl')

## RoBERTa

In [ ]:
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v3.pkl')
print(df.shape)
df.head()

In [ ]:
# 사용자, 아이템별 리뷰 집합 생성
user_review_set = df.groupby('user_id')['text'].apply(lambda row: ' '.join(row)).reset_index()
item_review_set = df.groupby('asin')['text'].apply(lambda row: ' '.join(row)).reset_index()

# column 변경
user_review_set.columns = ['user_id', 'user_text']
item_review_set.columns = ['asin', 'item_text']

# 결과 확인
print(user_review_set.shape)
print(item_review_set.shape)

user_review_set.head()

In [ ]:
item_review_set.head()

In [ ]:
# user, item 수 확인
user_review_set.shape, item_review_set.shape

In [ ]:
# load roberta
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModel.from_pretrained("roberta-base")

In [ ]:
# 사용자 리뷰 집합에서 bert embedding 추출 후 concat
user_text_embeddings = extract_cls_embeddings(user_review_set['user_text'].tolist(), tokenizer, model, batch_size=16)
user_text_embeddings = pd.DataFrame({'user_roberta_emb': [e for e in user_text_embeddings]})
user_review_set = pd.concat([user_review_set, user_text_embeddings], axis=1)

In [ ]:
# 아이템 리뷰 집합에서 bert embedding 추출 후 concat
item_text_embeddings = extract_cls_embeddings(item_review_set['item_text'].tolist(), tokenizer, model, batch_size=16)
item_text_embeddings = pd.DataFrame({'item_roberta_emb': [e for e in item_text_embeddings]})
item_review_set = pd.concat([item_review_set, item_text_embeddings], axis=1)

In [ ]:
# 뽑은 aspect를 join
df = pd.merge(df, user_review_set, on='user_id')
df = pd.merge(df, item_review_set, on='asin')
df= df.drop(columns=['user_text', 'item_text'])

In [ ]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v4.pkl')

In [ ]:
df.to_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v4.json')